# Stage 11 — Structure-Conditioned RBP Redesign (SageMaker, clean run)

Self-contained from a wild-type strict-CSV seed. Default: `Klebsiella → Enterobacter / UOX38086.1`.

**Pipeline:** 11a (ESMFold seed + Baseline Qualification gate) → 11b (ESM-IF1 beam search) → 11c (two-pass diversity prefilter) → 11d (08a validator, subprocess) → 11e (comparative report).

**Tested on:** `ml.g5.xlarge` / `2xlarge`, 90 GB EBS.

**Assumes** the corrected `08a_structural_fasttrack_validation.py`, `stage11_utils.py`, and `11b_run_inverse_folding_beam_search.py` are already in the zip (scale fixes, date-only run name, torch-scatter shim baked in).

## 1 — Unzip, fix Windows paths, install package, install runtime deps, patch fair-esm

One-shot. Idempotent. Survives kernel restarts.

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path

HOME = Path('/home/sagemaker-user')
ZIP  = HOME / 'phageforge_stage11.zip'
ROOT = HOME / 'phageforge_clean'

# --- 1a. Unzip if needed --------------------------------------------------
if not (ROOT / 'pyproject.toml').exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    assert ZIP.exists(), f'Upload the zip to {ZIP} first.'
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)

# --- 1b. Repair Windows-style backslash paths ----------------------------
# Windows-created zips dump files as `phageforge\__init__.py` on Linux extraction.
# Rebuild the directory tree.
for p in list(ROOT.iterdir()):
    if '\\' not in p.name:
        continue
    if p.name.endswith('\\'):
        p.unlink(missing_ok=True); continue
    target = ROOT.joinpath(*p.name.split('\\'))
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(p), str(target))

# Move stray data assets to expected locations
for src, dst in {
    'rbp_dataset_eskapee_strict.csv': 'data/processed/rbp_dataset_eskapee_strict.csv',
    'esm2_embeddings.pt':             'data/processed/strict/esm2_embeddings.pt',
    'esm2_embeddings_index.csv':      'data/processed/strict/esm2_embeddings_index.csv',
    'model.joblib':                   'results/broad/linear_probe/seed_42/model.joblib',
    'label_classes.json':             'results/broad/linear_probe/seed_42/label_classes.json',
}.items():
    s, d = ROOT / src, ROOT / dst
    if s.exists() and not d.exists():
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(s), str(d))
shutil.rmtree(ROOT / 'phageforge.egg-info', ignore_errors=True)
for pc in ROOT.rglob('__pycache__'):
    shutil.rmtree(pc, ignore_errors=True)

# --- 1c. Set CWD and env -------------------------------------------------
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
print('cwd:', os.getcwd())

# --- 1d. Install the phageforge package + runtime deps ------------------
PY = sys.executable
subprocess.run([PY, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-q',
                'fair-esm', '-U', 'biotite', 'torch-geometric'], check=True)

# --- 1e. Patch the installed fair-esm library (cannot ship this in our repo) -
import esm as _esm
ESM_UTIL = Path(_esm.__file__).parent / 'inverse_folding' / 'util.py'
if ESM_UTIL.exists():
    txt = ESM_UTIL.read_text()
    # (i) Biotite API rename: filter_backbone -> filter_peptide_backbone
    txt = txt.replace(
        'from biotite.structure import filter_backbone',
        'from biotite.structure import filter_peptide_backbone as filter_backbone',
    )
    # (ii) GPU->numpy safety: ensure .cpu() before .numpy() everywhere (idempotent)
    txt = txt.replace('.numpy()', '.cpu().numpy()').replace('.cpu().cpu().numpy()', '.cpu().numpy()')
    ESM_UTIL.write_text(txt)
    print('[OK] fair-esm util.py patched (biotite alias + .cpu().numpy())')

import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('transformers', transformers.__version__)
print('✅ environment ready')

## 2 — Run parameters

Change `SEED_PROTEIN_ID`, `SOURCE_HOST`, `TARGET_HOST`, and the 11b hyperparameters here. Everything downstream derives from this cell.

In [ ]:
import datetime

SEED_PROTEIN_ID = 'UOX38086.1'
SOURCE_HOST     = 'Klebsiella'
TARGET_HOST     = 'Enterobacter'
SEED            = 42
REUSE_CACHED_EMBEDDINGS = False

# --- 11b search hyperparameters ---
MIN_MUT, MAX_MUT = 4, 10
ROUNDS           = 6
BEAM_WIDTH       = 16
PROPOSALS        = 10

# --- Composite-score weights ---------------------------------------------------------------
# The values below are the canonical Stage 10/11 defaults; leaving them unchanged makes 11b
# use the exact built-in formula. Change any of them to ablate weights for a run — 11b applies
# them automatically. They map 1:1 to 11b's --w_target/--w_if1/--w_family/--w_identity/--w_mut_penalty parsers.
W_TARGET         = 0.30   # host-probability weight
W_IF1            = 0.45   # ESM-IF1 structural log-likelihood weight
W_FAMILY         = 0.15   # family-cosine weight
W_IDENTITY       = 0.10   # seed-identity weight
W_MUT_PENALTY    = 0.10   # mutation-count penalty (subtracted)

STRICT_CSV       = ROOT / 'data/processed/rbp_dataset_eskapee_strict.csv'
STRICT_EMB_PT    = ROOT / 'data/processed/strict/esm2_embeddings.pt'
STRICT_EMB_IDX   = ROOT / 'data/processed/strict/esm2_embeddings_index.csv'
PREDICTOR_MODEL  = ROOT / 'results/broad/linear_probe/seed_42/model.joblib'
PREDICTOR_LABELS = ROOT / 'results/broad/linear_probe/seed_42/label_classes.json'
EMBEDDING_MODEL  = 'facebook/esm2_t33_650M_UR50D'

for p in [STRICT_CSV, STRICT_EMB_PT, STRICT_EMB_IDX, PREDICTOR_MODEL, PREDICTOR_LABELS]:
    assert p.exists(), f'Missing required asset: {p}'
    print('✓', p)

DATE_TAG = datetime.datetime.now().strftime('%Y%m%d')
RUN_NAME = f"{SOURCE_HOST}_to_{TARGET_HOST}_{SEED_PROTEIN_ID.replace('.', '_')}_seed{SEED}_{DATE_TAG}"
RUN_DIR  = ROOT / 'results/stage11' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('\nrun_dir =', RUN_DIR)

## 3 — Stage 11a: context + ESMFold seed + Baseline Qualification gate

Exit codes: `0` ok | `1` input | `2` gate failed (pick another seed) | `3` inference.

In [ ]:
cmd = [
    sys.executable, 'scripts/11a_prepare_stage11_context.py',
    '--strict_csv', str(STRICT_CSV),
    '--seed_protein_id', SEED_PROTEIN_ID,
    '--source_host', SOURCE_HOST,
    '--target_host', TARGET_HOST,
    '--out_dir', str(RUN_DIR),
    '--run_name', RUN_NAME,
    '--strict_embeddings', str(STRICT_EMB_PT),
    '--strict_embeddings_index', str(STRICT_EMB_IDX),
    '--predictor_model', str(PREDICTOR_MODEL),
    '--predictor_label_classes', str(PREDICTOR_LABELS),
    '--embedding_model', EMBEDDING_MODEL,
    '--esmfold_device', 'cuda', '--esmfold_chunk_size', '128', '--esmfold_num_recycles', '1',
    '--min_seed_plddt', '70.0',
    '--family_top_n', '32', '--target_top_m', '8',
    '--family_cosine_floor', '0.85', '--length_tolerance', '0.05',
    '--max_edit_positions', '8', '--soft_positions', '3',
    '--min_mutations', str(MIN_MUT), '--max_mutations', str(MAX_MUT),
    '--entropy_floor', '0.20', '--family_top_k', '4', '--target_top_k', '4',
    '--max_allowed_aas_per_pos', '6', '--region_block', '50',
    '--seed', str(SEED),
]
if REUSE_CACHED_EMBEDDINGS:
    cmd.append('--reuse_cached_embeddings')

rc = subprocess.run(cmd).returncode
print('\n11a exit:', rc)
if rc == 2:
    raise SystemExit('Baseline Qualification FAILED — pick a different SEED_PROTEIN_ID and rerun.')
assert rc == 0, '11a failed.'

CTX_JSON = RUN_DIR / 'context' / 'stage11_context.json'
SEED_PDB = RUN_DIR / 'context' / 'seed_wt.pdb'
print('context:', CTX_JSON.exists(), '|', CTX_JSON)
print('seed PDB:', SEED_PDB.exists(), '|', SEED_PDB)

## 4 — Stage 11b: inverse-folding beam search

Composite: `0.30·target + 0.45·if1 + 0.15·family + 0.10·identity − 0.10·mut_penalty` by default. We override `w_target=0.60`, `w_if1=0.30`, `w_mut_penalty=0.005` to push target probability while staying inside the structural manifold.

In [ ]:
import pandas as pd
SEARCH_CSV  = RUN_DIR / 'search' / 'stage11_search_candidates.csv'
SEARCH_JSON = RUN_DIR / 'search' / 'stage11_search_summary.json'

cmd = [
    sys.executable, 'scripts/11b_run_inverse_folding_beam_search.py',
    '--stage11_context_json', str(CTX_JSON),
    '--predictor_model', str(PREDICTOR_MODEL),
    '--label_classes_json', str(PREDICTOR_LABELS),
    '--out_csv', str(SEARCH_CSV),
    '--out_json', str(SEARCH_JSON),
    '--embedding_model', EMBEDDING_MODEL,
    '--if_device', 'cuda', '--if_chain_id', 'A',
    '--rounds', str(ROUNDS), '--beam_width', str(BEAM_WIDTH),
    '--proposals_per_parent', str(PROPOSALS), '--substitutions_per_position', '4',
    '--batch_size', '4', '--seed', str(SEED),
    '--w_target', str(W_TARGET),
    '--w_if1', str(W_IF1),
    '--w_family', str(W_FAMILY),
    '--w_identity', str(W_IDENTITY),
    '--w_mut_penalty', str(W_MUT_PENALTY),
]
rc = subprocess.run(cmd).returncode
print('11b exit:', rc); assert rc == 0, f'11b failed with exit code {rc}'
pd.read_csv(SEARCH_CSV).head()

## 5 — Stage 11c: two-pass diversity prefilter (top-10 → top-3)

In [ ]:
TOP10_CSV = RUN_DIR / 'prefilter' / 'stage11_top10.csv'
TOP3_CSV  = RUN_DIR / 'prefilter' / 'stage11_top3.csv'
PRE_JSON  = RUN_DIR / 'prefilter' / 'stage11_prefilter_summary.json'

cmd = [
    sys.executable, 'scripts/11c_prefilter_stage11_candidates.py',
    '--stage11_context_json', str(CTX_JSON),
    '--search_csv', str(SEARCH_CSV),
    '--out_topk_csv', str(TOP10_CSV),
    '--out_topk_final_csv', str(TOP3_CSV),
    '--out_json', str(PRE_JSON),
    '--top_k', '10', '--top_k_final', '3',
    '--embedding_model', EMBEDDING_MODEL,
    '--batch_size', '4', '--seed', str(SEED),
]
rc = subprocess.run(cmd).returncode
print('11c exit:', rc); assert rc == 0
pd.read_csv(TOP3_CSV)

## 6 — Stage 11d: 08a structural validator (subprocess; top-10 + top-3)

Validator is the patched 08a (pLDDT now on 0–100 scale, matching the 70.0 gate).

In [ ]:
VAL10_DIR = RUN_DIR / 'validation_top10'
VAL3_DIR  = RUN_DIR / 'validation_top3'
for d in (VAL10_DIR, VAL3_DIR):
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

def _val(panel_csv, out_dir, top_k, label):
    out_json = out_dir / f'stage11_top{top_k}_launch.json'
    cmd = [
        sys.executable, 'scripts/11d_validate_stage11_candidates.py',
        '--validated_csv', str(panel_csv), '--ranked_csv', str(panel_csv),
        '--context_json', str(CTX_JSON),
        '--validator_script', 'scripts/08a_structural_fasttrack_validation.py',
        '--out_dir', str(out_dir), '--out_json', str(out_json),
        '--top_k', str(top_k),
        '--device', 'cuda', '--chunk_size', '128', '--num_recycles', '1',
    ]
    print(f'\n→ Validating {label} …')
    rc = subprocess.run(cmd).returncode
    assert rc == 0, f'11d ({label}) failed.'

_val(TOP10_CSV, VAL10_DIR, 10, 'top-10')
_val(TOP3_CSV,  VAL3_DIR,   3, 'top-3')

VAL_CSV = VAL3_DIR / 'stage08_structural_fasttrack_summary.csv'
cols = ['sample_id','esmfold_mean_plddt','mutation_site_mean_plddt',
        'rmsd_to_selected_seed','mutation_site_confidence_ge70_fraction',
        'stage08_pass','stage08_decision_reason']
pd.read_csv(VAL_CSV)[cols]

## 7 — Stage 11e: comparative report (vs historical Stage 08/09/10)

In [ ]:
REPORT_DIR = RUN_DIR / 'report'
BASE_08 = ROOT / 'results/stage08/structural_fasttrack/stage08_structural_fasttrack_summary.csv'
BASE_09 = ROOT / 'results/stage09/validation_top3/stage08_structural_fasttrack_summary.csv'
BASE_10 = ROOT / 'results/stage10/validation_top3_pdbs/stage08_structural_fasttrack_summary.csv'

cmd = [
    sys.executable, 'scripts/11e_make_stage11_report.py',
    '--stage11_context_json', str(CTX_JSON),
    '--search_csv', str(SEARCH_CSV),
    '--prefilter_csv', str(TOP10_CSV),
    '--validation_csv', str(VAL_CSV),
    '--out_dir', str(REPORT_DIR),
]
for flag, val in (('--baseline_stage08_csv', BASE_08),
                  ('--baseline_stage09_csv', BASE_09),
                  ('--baseline_stage10_csv', BASE_10)):
    if val.exists(): cmd += [flag, str(val)]

rc = subprocess.run(cmd).returncode
print('11e exit:', rc); assert rc == 0

from IPython.display import Markdown, display
display(Markdown((REPORT_DIR / 'stage11_report.md').read_text()))

## 8 — Package outputs for download

In [ ]:
import tarfile
archive = ROOT / f'stage11_{RUN_NAME}.tar.gz'
targets = {
    'context':           RUN_DIR / 'context',
    'search':            RUN_DIR / 'search',
    'prefilter':         RUN_DIR / 'prefilter',
    'validation_top10':  VAL10_DIR,
    'validation_top3':   VAL3_DIR,
    'report':            REPORT_DIR,
    'run_metadata.json': RUN_DIR / 'run_metadata.json',
}
with tarfile.open(archive, 'w:gz') as tar:
    for arc, path in targets.items():
        if path.exists():
            print(' →', path.relative_to(ROOT)); tar.add(path, arcname=arc)
        else:
            print(' [!] skip (missing):', path)
mb = archive.stat().st_size / (1024 * 1024)
print(f'\n✅ Archive: {archive.name}  ({mb:.2f} MB)')
print('Right-click it in the JupyterLab file pane → Download.')